In [2]:
# ============================================================
# FILE: utils/preprocessing.py
# PURPOSE: Load, clean, and encode the dataset
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle
import os


In [3]:
# ── 1. Load Dataset ──────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"✅ Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"   Columns: {list(df.columns)}")
    return df

In [4]:
# ── 2. Basic Cleaning ─────────────────────────────────────────
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # Drop duplicates
    before = len(df)
    df = df.drop_duplicates()
    print(f"🧹 Removed {before - len(df)} duplicate rows")

    # Drop Student_ID — it's an identifier, not a feature
    if "Student_ID" in df.columns:
        df = df.drop(columns=["Student_ID"])

    # Strip whitespace from string columns
    str_cols = df.select_dtypes(include="object").columns
    for col in str_cols:
        df[col] = df[col].str.strip()

    # Handle missing values
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    print(f"✅ Cleaned data shape: {df.shape}")
    return df




In [5]:
# ── 3. Encode Categorical Columns ────────────────────────────
def encode_data(df: pd.DataFrame, save_dir: str = "Project File/"):
    """
    Label-encode all object columns.
    Saves encoders to disk so Streamlit can reuse them at prediction time.
    Returns: X (features), y (target), feature_names, label_encoders dict
    """
    os.makedirs(save_dir, exist_ok=True)
    label_encoders = {}
    df_encoded = df.copy()

    cat_cols = df.select_dtypes(include="object").columns.tolist()
    # Remove target from cat_cols if present
    if "Career" in cat_cols:
        cat_cols.remove("Career")

    for col in cat_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le

    # Encode target
    le_target = LabelEncoder()
    df_encoded["Career"] = le_target.fit_transform(df_encoded["Career"])
    label_encoders["Career"] = le_target

    # Save encoders
    with open(os.path.join(save_dir, "label_encoders.pkl"), "wb") as f:
        pickle.dump(label_encoders, f)
    print(f"✅ Saved label encoders → {save_dir}label_encoders.pkl")

    # Split features / target
    X = df_encoded.drop(columns=["Career"])
    y = df_encoded["Career"]
    feature_names = list(X.columns)

    return X, y, feature_names, label_encoders




In [6]:
# ── 4. Scale Numerical Features ──────────────────────────────
def scale_features(X_train, X_test, save_dir: str = "Project File/"):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    with open(os.path.join(save_dir, "scaler.pkl"), "wb") as f:
        pickle.dump(scaler, f)
    print(f"✅ Saved scaler → {save_dir}scaler.pkl")

    return X_train_scaled, X_test_scaled, scaler




In [7]:
# ── Quick test ────────────────────────────────────────────────
if __name__ == "__main__":
    DATA_PATH = r"C:\Users\LENOVO\OneDrive\Desktop\Career Mentor\Project File\career_dataset_realistic.csv"
    df = load_data(DATA_PATH)
    df = clean_data(df)
    print("\n📊 Value counts for target (Career):")
    print(df["Career"].value_counts())
    print("\n📊 Data types after cleaning:")
    print(df.dtypes)

✅ Loaded dataset: 1010 rows, 11 columns
   Columns: ['Student_ID', 'Interest', 'Primary_Skill', 'Coding_Score', 'Math_Score', 'Communication_Score', 'Projects_Count', 'Certifications_Count', 'GitHub_Score', 'Personality_Type', 'Career']
🧹 Removed 0 duplicate rows
✅ Cleaned data shape: (1010, 10)

📊 Value counts for target (Career):
Career
Software Engineer            275
Data Scientist               142
Security Analyst             139
Cloud Engineer               124
Frontend Developer           112
Backend Developer             64
Data Analyst                  56
DevOps Engineer               56
Ethical Hacker                29
Machine Learning Engineer     13
Name: count, dtype: int64

📊 Data types after cleaning:
Interest                object
Primary_Skill           object
Coding_Score             int64
Math_Score               int64
Communication_Score      int64
Projects_Count           int64
Certifications_Count     int64
GitHub_Score             int64
Personality_Type        o